In [ ]:
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def gpu_mem_used_gb(device: int = 0) -> float:
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)

# Llama3.1-8B-PRM-Deepseek-Data Scoring Smoke Test

Minimal notebook for scoring the same flamingo toy example with
`RLHFlow/Llama3.1-8B-PRM-Deepseek-Data`. This PRM is a causal LM
trained to judge each step by predicting `+` or `-` in an assistant
turn.

In [ ]:
# Model paths
base_dir = "/groups/chichengz/tnn/datasets"
llama_prm_dir = f"{base_dir}/Llama3.1-8B-PRM-Deepseek-Data"

In [ ]:
# Same toy example used in the Qwen PRM smoke test.
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [ ]:
# Llama 8B PRM fits comfortably on a 32 GB V100 in fp16.
llama_tokenizer = AutoTokenizer.from_pretrained(llama_prm_dir)
llama_model = AutoModelForCausalLM.from_pretrained(
    llama_prm_dir,
    device_map="cuda:0",
    torch_dtype=torch.float16,
).eval()

llama_tokenizer.padding_side = "right"
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_model.config.pad_token_id = llama_model.config.eos_token_id

plus_token_id = llama_tokenizer.encode("+")[-1]
minus_token_id = llama_tokenizer.encode("-")[-1]
candidate_token_ids = [plus_token_id, minus_token_id]

print(f"plus_token_id : {plus_token_id}")
print(f"minus_token_id: {minus_token_id}")
print(f"dtype         : {next(llama_model.parameters()).dtype}")
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

In [ ]:
def score_llama_prm(model, tokenizer, problem, steps, candidate_token_ids):
    """Return per-step P(+) / (P(+) + P(-)) for RLHFlow's Llama PRM.

    The conversation alternates user reasoning-step messages with
    assistant `+` judgements. For each prefix we score the token
    position immediately before the current assistant `+` token.
    """
    marker_text = "\u043a\u0438"  # "ки" — Cyrillic, unique in the vocab
    marker_token_id = (
        tokenizer(marker_text, return_tensors="pt").input_ids[0, 1].item()
    )

    conversation = []
    marker_conversation = []
    step_scores = []

    for step_idx, step in enumerate(steps):
        text = (problem + " " + step) if step_idx == 0 else step

        conversation.append({"role": "user", "content": text})
        conversation.append({"role": "assistant", "content": "+"})

        marker_conversation.append({"role": "user", "content": text})
        marker_conversation.append(
            {"role": "assistant", "content": marker_text}
        )

        input_ids = tokenizer.apply_chat_template(
            conversation,
            return_tensors="pt",
        ).to(model.device)
        marker_input_ids = tokenizer.apply_chat_template(
            marker_conversation,
            return_tensors="pt",
        ).to(model.device)

        if input_ids.shape != marker_input_ids.shape:
            raise RuntimeError(
                f"Marker conversation shape mismatch: "
                f"{input_ids.shape} vs {marker_input_ids.shape}"
            )

        with torch.no_grad():
            logits = model(input_ids=input_ids).logits[
                :, :, candidate_token_ids
            ]
            probs = logits.softmax(dim=-1)[:, :, 0]

        # The model predicts token N from position N-1. Locate the
        # marker token in the parallel prompt and read the previous
        # position's P(+). Use the last marker for the current step.
        marker_positions = (
            marker_input_ids[0, 1:] == marker_token_id
        ).nonzero(as_tuple=True)[0]
        if marker_positions.numel() != step_idx + 1:
            raise RuntimeError(
                f"Expected {step_idx + 1} marker positions, found "
                f"{marker_positions.numel()}"
            )
        score_pos = marker_positions[-1].item()
        step_scores.append(
            probs[0, score_pos].detach().cpu().float().item()
        )

    return step_scores


llama_scores = score_llama_prm(
    llama_model,
    llama_tokenizer,
    problem,
    steps,
    candidate_token_ids,
)

for i, (step, score) in enumerate(zip(steps, llama_scores), start=1):
    print(f"step {i}: P(correct) = {score:.4f}")
    print(step)
    print()

In [ ]:
# Free Llama PRM.
del llama_model, llama_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")